In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master('local[*]').appName('test').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/25 23:10:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df_green = spark.read.option('recursiveFileLookup', 'true').parquet('./data/pq/green')

In [5]:
df_green.show(5)

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|       2| 2020-01-23 21:38:32|  2020-01-23 21:47:38|                 N|         1|         145|         179|              1|         2.35|        9.5|  0.5|    0.

In [6]:
rdd = df_green\
            .select('lpep_pickup_datetime', 'PULocationID', 'total_amount')\
            .rdd

In [7]:
rdd.take(5)

[Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 23, 21, 38, 32), PULocationID=145, total_amount=10.8),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 11, 8, 19, 49), PULocationID=82, total_amount=9.8),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 4, 22, 3, 14), PULocationID=83, total_amount=8.8),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 25, 18, 19, 19), PULocationID=7, total_amount=12.3),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 10, 8, 28, 18), PULocationID=196, total_amount=28.55)]

In [ ]:
from datetime import datetime

start = datetime(2020, 1, 1)

In [14]:
def filter_outliers(row):
    return row.lpep_pickup_datetime >= start

In [17]:
def prepare_for_grouping(row):
    hour = row.lpep_pickup_datetime.replace(minute = 0, second = 0, microsecond = 0)
    zone = row.PULocationID
    key = (hour, zone)

    amount = row.total_amount
    count = 1
    value = (amount, count)

    return (key, value)

In [27]:
def calculate_revenue(left, right):
    left_amount, left_count = left
    right_amount, right_count = right

    output_amount = left_amount + right_amount
    output_count = left_count + right_count

    return (output_amount, output_count)

In [45]:
from collections import namedtuple

RevenueRow = namedtuple('RevenueRow', ['hour', 'zone', 'revenue', 'count'])

In [46]:
def unwrap(row):
    return RevenueRow(row[0][0], row[0][1], row[1][0], row[1][1])

In [48]:
from pyspark.sql import types

result_schema = types.StructType([
    types.StructField('hour', types.TimestampType(), True),
    types.StructField('zone', types.IntegerType(), True),
    types.StructField('revenue', types.DoubleType(), True),
    types.StructField('count', types.IntegerType(), True),
])

df_result = rdd \
    .filter(filter_outliers)\
    .map(prepare_for_grouping)\
    .reduceByKey(calculate_revenue)\
    .map(unwrap)\
    .toDF(result_schema)

In [49]:
df_result.show()

+-------------------+----+------------------+-----+
|               hour|zone|           revenue|count|
+-------------------+----+------------------+-----+
|2020-01-10 08:00:00| 196|             40.35|    2|
|2020-01-20 08:00:00| 232|              22.0|    1|
|2020-01-24 11:00:00|  75| 846.7899999999998|   48|
|2020-01-07 22:00:00| 181| 484.2700000000001|   27|
|2020-01-08 17:00:00| 130| 517.0800000000002|   28|
|2020-01-21 11:00:00| 205|            138.07|    3|
|2020-01-27 19:00:00|  25|            481.29|   28|
|2020-01-13 17:00:00| 244| 820.2699999999998|   30|
|2020-01-22 20:00:00|  37|            110.95|    4|
|2020-01-04 09:00:00|   7|260.43000000000006|   21|
|2020-01-08 13:00:00|  65|             268.0|   16|
|2020-01-06 07:00:00|  42| 551.1200000000002|   37|
|2020-01-05 17:00:00|  25|            217.37|   18|
|2020-01-22 09:00:00| 212|            182.94|    5|
|2020-01-31 23:00:00| 166|210.13000000000008|   17|
|2020-01-02 11:00:00| 108|            139.13|    4|
|2020-01-08 

Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/Applications/miniconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe


In [50]:
df_result.write.parquet('./data/tmp/green-revenue')

25/07/25 23:56:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


In [51]:
df_result.rdd.getNumPartitions()

8